# 07. Feature Engineering V5 — 제품명 강조 문구

**목적**  
제품명에 강조된 보습·진정·장벽 등의 주제를 특징으로 변환합니다.

**입력**  
`data/interim/전성분_정규화_세로형.csv`

**출력**  
`data/interim/V5_제품명_강조주제.csv`

> 저장된 전처리 데이터만 사용하며 외부 요청은 발생하지 않습니다.


In [ ]:
from pathlib import Path

# Jupyter와 Colab 모두 저장소 루트에서 실행합니다.
def find_project_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "data").is_dir() and (path / "notebooks").is_dir():
            return path
    raise FileNotFoundError("저장소를 clone한 뒤 해당 폴더 안에서 실행하세요.")

PROJECT_ROOT = find_project_root()

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

for directory in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


## V5 - 제품명 강조 문구 (Feature Engineering)

이 노트북은 다음 파일 3개를 만듭니다.
1. V5_제품명_강조주제.csv        : 제품별 강조 테마(다중라벨) + 판정 근거
2. V5_제품별_강조주제.csv: 테마별 제품수 + 해당 제품 세로 나열
3. V5_주제별_성분비율.csv: 테마별 성분 함유 비율


In [ ]:
import os
import re
import pandas as pd

INPUT_PATH = DATA_INTERIM_DIR / "전성분_정규화_세로형.csv"
OUTPUT_DIR = DATA_INTERIM_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
df = pd.read_csv(INPUT_PATH)
name_map = df[['product_id', 'product_name_raw', 'product_name_clean']].drop_duplicates(subset='product_id')

### 테마 키워드와 판촉 문구 제외 규칙

제품명의 보습·진정·장벽 등 주제 키워드를 분류하고, 대괄호 안의 세일·이벤트·홍보 문구는 분석에서 제외합니다.


In [4]:
THEME_KEYWORDS = {
    '수분·수부지·아쿠아': ['수분','수부지','아쿠아','워터','히알루','하이알','하이드로','하이드레이팅','수분잠금','속보습','수분충전','물착','수분광채'],
    '보습·장벽·리페어': ['보습','장벽','베리어','배리어','리페어','아토','모이스처','모이스춰','크림밤','멀티밤','베리어덤','세라마이드','세라','판테놀','판토','밀착보습막','손상장벽','손상/장벽','보습장벽','디판테놀'],
    '진정·시카·민감': ['진정','시카','센텔라','마데카','카밍','수딩','민감','레드','붉은기','흔적','블레미쉬','알란토인','테카','카밍다운'],
    '피지·모공·트러블': ['피지','모공','트러블','여드름','아크네','세비엄','세보','포어','타이트','모공수축','블레미쉬','AC','에이클리어','테라크네','노스카나인'],
    '탄력·주름·안티에이징': ['탄력','주름','리프팅','안티','레티놀','레티날','콜라겐','PDRN','피디알엔','EGF','리쥬','바운스','리모델링','밀도','광채탄력','목주름','타임리버스','비타리프트'],
    '미백·잡티·톤업': ['미백','잡티','톤업','브라이트','기미','색소','멜라','글루타치온','비타','비타민','토닝','광채','화이트','기미톤업','비타C','톤프렙'],
    '남성·포맨': ['포 맨','포맨','남성'],
}

PROMO_PATTERNS = [
    r'PICK', r'Pick', r'pick', r'기획', r'증정', r'단품', r'대용량',
    r'^NEW$', r'^\d+\+\d+$', r'올영픽', r'올영\s?\d', r'한정', r'특가',
    r'\d+\s*ml', r'어워즈',
]
promo_regex = re.compile('|'.join(PROMO_PATTERNS))
date_pattern = re.compile(r'^\d{1,2}/\d{1,2}')

In [5]:
def extract_raw_keywords(raw_name):
    """product_name_raw 대괄호 태그에서 판촉 문구를 제외한 나머지만 남김"""
    bracket_contents = re.findall(r'\[([^\]]+)\]', str(raw_name))
    keywords = []
    for content in bracket_contents:
        content = content.strip()
        if date_pattern.match(content):
            continue  # "7/21하루특가"처럼 날짜로 시작하는 세일 태그는 통째로 제외
        for token in re.split(r'[/|]', content):
            token = token.strip()
            if token and not promo_regex.search(token):
                keywords.append(token)
    return keywords

In [6]:
def find_matches(text):
    """텍스트 안에서 테마별로 실제 매칭된 키워드를 같이 반환 (근거 추적용)"""
    text = str(text)
    matches = {}
    for theme, kws in THEME_KEYWORDS.items():
        hit = [kw for kw in kws if kw in text]
        if hit:
            matches[theme] = hit
    return matches

In [7]:
def classify_row(row):
    """이중 체크: product_name_clean 우선 매칭, 실패 시 raw 필터링 키워드로 재시도"""
    clean_matches = find_matches(row['product_name_clean'])
    if clean_matches:
        themes = list(clean_matches.keys())
        clean_predict = '; '.join(f'{t}({",".join(kws)})' for t, kws in clean_matches.items())
        return themes, clean_predict, ''  # clean에서 성공했으므로 raw는 검사 안 함

    raw_keywords = extract_raw_keywords(row['product_name_raw'])
    raw_matches = find_matches(' '.join(raw_keywords))
    if raw_matches:
        themes = list(raw_matches.keys())
        raw_predict = '; '.join(f'{t}({",".join(kws)})' for t, kws in raw_matches.items())
        return themes, '', raw_predict

    return ['기타/강조문구 약함'], '', ''

### 파일 1: 제품별 강조 테마 (다중라벨, wide 형태) + 판정 근거


In [ ]:
results = name_map.apply(classify_row, axis=1, result_type='expand')
results.columns = ['themes', 'clean_predict', 'raw_predict']
name_map = pd.concat([name_map.reset_index(drop=True), results.reset_index(drop=True)], axis=1)

max_themes = name_map['themes'].apply(len).max()
for i in range(max_themes):
    name_map[f'강조문구_{i+1}'] = name_map['themes'].apply(lambda t, i=i: t[i] if i < len(t) else '')

theme_cols = [f'강조문구_{i+1}' for i in range(max_themes)]
file1 = name_map[['product_id', 'product_name_raw', 'product_name_clean'] + theme_cols + ['clean_predict', 'raw_predict']]
file1.to_csv(os.path.join(OUTPUT_DIR, 'V5_제품명_강조주제.csv'), index=False, encoding='utf-8-sig')

### 파일 2: 테마별 제품수 + 제품 세로 나열


In [9]:
ALL_THEMES = list(THEME_KEYWORDS.keys()) + ['기타/강조문구 약함']
theme_counts = {t: name_map['themes'].apply(lambda ts, t=t: t in ts).sum() for t in ALL_THEMES}
sorted_themes = sorted(theme_counts.items(), key=lambda x: -x[1])

rows2 = []
for theme, count in sorted_themes:
    subset = name_map[name_map['themes'].apply(lambda ts, theme=theme: theme in ts)][['product_name_raw', 'product_name_clean']]
    for i, (_, r) in enumerate(subset.iterrows()):
        rows2.append({
            '테마': theme if i == 0 else '',
            '제품수': count if i == 0 else '',
            'product_name_raw': r['product_name_raw'],
            'product_name_clean': r['product_name_clean'],
        })
file2 = pd.DataFrame(rows2)
file2.to_csv(os.path.join(OUTPUT_DIR, 'V5_제품별_강조주제.csv'), index=False, encoding='utf-8-sig')

### 파일 3: 테마별 성분 함유 비율

각 테마에 속한 제품들의 전체 성분(canonical_name, V1과 동일 기준)을 모아서,
성분별로 그 테마 안에서 몇 개 제품에 들어있는지/비율을 계산.


In [ ]:
product_ingredients = df.groupby('product_id')['canonical_name'].apply(lambda x: set(x.dropna()))

rows3 = []
for theme in ALL_THEMES:
    theme_pids = name_map[name_map['themes'].apply(lambda ts, theme=theme: theme in ts)]['product_id']
    total_in_theme = len(theme_pids)
    if total_in_theme == 0:
        continue
    ing_counter = {}
    for pid in theme_pids:
        for ing in product_ingredients.get(pid, set()):
            ing_counter[ing] = ing_counter.get(ing, 0) + 1
    for ing, cnt in ing_counter.items():
        rows3.append({
            'thema': theme,
            'canonical_name': ing,
            '제품에_함유된_개수': cnt,
            '테마별_전체_제품수': total_in_theme,
            '함유비율': round(cnt / total_in_theme, 3),
        })

file3 = pd.DataFrame(rows3).sort_values(['thema', '함유비율'], ascending=[True, False])
file3.to_csv(os.path.join(OUTPUT_DIR, 'V5_주제별_성분비율.csv'), index=False, encoding='utf-8-sig')

print('파일1:', file1.shape, '| 파일2:', file2.shape, '| 파일3:', file3.shape)
print('저장 완료:', OUTPUT_DIR)